<a href="https://colab.research.google.com/github/jeevanshrestha/GenAi/blob/main/RAG_with_OpenAI_File_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG with OpenAI File Search

### Initial Setup

#### Install Required Libraries


In [1]:
!pip install openai langchain langchain-community langchain_huggingface langchain-openai unstructured faiss-cpu grandalf  -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 11.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.6/167.6 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.8/195.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━

#### Mount Google Drive

In [4]:
# Mount Google Drive to access files
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
#set working directory
import os
os.chdir('/content/drive/MyDrive/GenAI/RAG/')

In [36]:
directory = os.getcwd()  # Get current working directory
file_directory = os.path.join(directory, 'RAG with OpenAI File Search (1)')  # Target subdirectory

# Define full paths to two files inside that directory
file_paths = [
    os.path.join(file_directory, 'Termos e Condições _ Politica de Privacidade da Bitte (PT_EN).docx'),
    os.path.join(file_directory, 'Bitte - Apresentação Corporativa.pdf')
]

print(repr(file_paths[0]))
print(repr(file_paths[1]))
for path in file_paths:
    if os.path.exists(path):
        print(f"✅ File exists: {path}")
    else:
        print(f"❌ File missing: {path}")

'/content/drive/MyDrive/GenAI/RAG/RAG with OpenAI File Search (1)/Termos e Condições _ Politica de Privacidade da Bitte (PT_EN).docx'
'/content/drive/MyDrive/GenAI/RAG/RAG with OpenAI File Search (1)/Bitte - Apresentação Corporativa.pdf'
✅ File exists: /content/drive/MyDrive/GenAI/RAG/RAG with OpenAI File Search (1)/Termos e Condições _ Politica de Privacidade da Bitte (PT_EN).docx
✅ File exists: /content/drive/MyDrive/GenAI/RAG/RAG with OpenAI File Search (1)/Bitte - Apresentação Corporativa.pdf


#### OpenAI Key Setup

In [37]:
from google.colab import userdata
api_key = userdata.get('genai_course')

#### Import Libraries


In [38]:
#Import the libraries
from langchain_openai import ChatOpenAI
from IPython.display import display, Markdown
import json
import numpy as np
import pandas as pd

In [39]:
print(os.getcwd())

/content/drive/MyDrive/GenAI/RAG


## Create Knowledge Component

In [41]:
from openai import OpenAI # must be above 1.66
from IPython.display import Markdown

client = OpenAI(api_key=api_key)

### Upload file to OpenAPI file storage

In [42]:
file_ids = []

#Upload the files to the API
for file_path in file_paths:
    with open(file_path, "rb") as file:
        response = client.files.create(
            file=file,
            purpose='assistants'
        )
        file_ids.append(response.id)


In [43]:
file_ids

['file-7vjMKSkj4izbXFr8mQwqwc', 'file-NwLnn9Vjpcsvh9z4Vjhc68']

### Convert the files to vector store

In [44]:
# Create a Vector Store
vector_store = client.vector_stores.create(
    name = "Bitte Vector Store",
)
print(vector_store.id)

vs_68536eb03604819190801ce9df9191cd


In [47]:
# Add files to the vector store
for file_id in file_ids:
  result = client.vector_stores.files.create(
      vector_store_id = vector_store.id,
      file_id = file_id,
  )

## RAG SYSTEM

In [60]:
# Build the RAG system
response = client.responses.create(
    model = "gpt-4.1-mini",
    input = "List the benefits of Bitte in English",
    tools = [{
        "type": "file_search",
        "vector_store_ids": [vector_store.id],
        "max_num_results": 20
    }],
    include = ["file_search_call.results"] # Include the references
)

In [61]:
response

Response(id='resp_685373afb7d88198a26928d02e9f61230214231a45f3b565', created_at=1750299567.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4.1-mini-2025-04-14', object='response', output=[ResponseFileSearchToolCall(id='fs_685373b0cd188198ae9a7c4bece709ec0214231a45f3b565', queries=['benefits of Bitte'], status='completed', type='file_search_call', results=[Result(attributes={}, file_id='file-NwLnn9Vjpcsvh9z4Vjhc68', filename='Bitte - Apresentação Corporativa.pdf', score=0.0304, text='Commerce_aaS project  Kick-off meeting | First thoughts and hypothesis\n\n\nBitte - a \nsolução digital \npara o seu \nrestaurante\n\nBudget Allocation\n\n\n\nQue desafios sentem os restaurantes atualmente?\n\n1 Recrutamento e gestão de equipas\n\nOs restaurantes têm dificuldades em encontrar \n\ncolaboradores dedicados, dada a elevada rotação do \n\nstaff e a falta de mão de obra disponível.\n\n2 Expectativas dos clientes\n\nOs clientes modernos exigem experiências person

In [62]:
# What GPT queried the Vecto Store
response.output[0].queries

['benefits of Bitte']

In [63]:
# What was the context retrieved?
response.output[0].results[0].text

'Commerce_aaS project  Kick-off meeting | First thoughts and hypothesis\n\n\nBitte - a \nsolução digital \npara o seu \nrestaurante\n\nBudget Allocation\n\n\n\nQue desafios sentem os restaurantes atualmente?\n\n1 Recrutamento e gestão de equipas\n\nOs restaurantes têm dificuldades em encontrar \n\ncolaboradores dedicados, dada a elevada rotação do \n\nstaff e a falta de mão de obra disponível.\n\n2 Expectativas dos clientes\n\nOs clientes modernos exigem experiências personalizadas e \n\norientadas às suas preferências, um tratamento diferenciado \n\ne uma interação simplificada com os restaurantes que \n\nfrequentam.\n\n3 Eficiência e gestão\n\nOs restaurantes precisam de ferramentas para otimizar \n\noperações, gerir pedidos e reservas, e analisar dados \n\npara uma tomada de decisão mais eficaz e que traga \n\nmais lucro e melhores resultados..\n\n4 Digitalização e inovação\n\nA adoção de soluções digitais é crucial para se manter \n\ncompetitivo e alcançar o sucesso no mercado atua

In [64]:
# What is the answer?
Markdown(response.output[1].content[0].text)

The benefits of Bitte, a digital solution for restaurants, translated into English and summarized, are as follows:

1. **Improved Customer Experience**  
   - Provides a modern and personalized service.  
   - Customers can better know the menu with photos for an informed choice.  
   - Interaction is more attentive and focused on customer satisfaction because staff spend less time on operational tasks.

2. **More Sales Channels**  
   - Allows customers to place orders for home delivery or takeaway with ease.  
   - Brings more sales, more customers, and increased revenue.

3. **Ease of Use and Setup**  
   - No programming or technical knowledge needed to start.  
   - No need to install software on restaurant or customer devices.  
   - Menu structure is created quickly using OCR from a menu PDF—saving time.

4. **Efficient Staff and Operations**  
   - Digitalized processes make the team more efficient.  
   - Helps overcome challenges like high staff turnover by reducing operational tasks.  
   - The staff can focus more on personalized service.

5. **Data-Driven Management and Decision Making**  
   - Provides insights on the most profitable and popular dishes.  
   - Tracks preparation times, kitchen efficiency by day, and other valuable data.  
   - Supports serious, periodic evaluations of the menu to maintain profitability.

6. **Digital Innovation and Visibility**  
   - Helps restaurants stay competitive and innovative by adopting digital tools.  
   - Increases visibility to a demanding customer base even before they choose the restaurant.  
   - Expands offerings by enabling takeaway and delivery with simple, effective processes.

7. **Revenue Increase and Upselling Features**  
   - Digital menus can increase revenue by 10-20% compared to traditional menus.  
   - Attractive visual presentation and personalization encourage customers to spend more.  
   - Ability to suggest pairings (e.g., drinks or desserts matching dishes) to increase the average ticket per customer.

8. **Freemium Model with Upgrade Options**  
   - Free basic digital menu with capabilities such as campaign creation, multiple language translation, and cross-selling features.  
   - Premium subscription adds functionalities like mobile orders and payments, table reservations linked with Google, sending promotions, takeaway/delivery management, sales analytics, rentability analysis, popularity analysis, and more.

In summary, Bitte offers a comprehensive digital platform designed to enhance the restaurant customer experience, increase efficiency, improve sales through data insights, and simplify operations with innovative technology, all while being easy to start and use.

## RAG System with Developer Message

In [65]:
# Build the RAG system
response = client.responses.create(
    model = "gpt-4.1-mini",
    input = "List the benefits of Bitte in English",
    instructions = "Answer like a toxic CEO who prefers terms like pre-revenue and cash burn ratio",
    tools = [{
        "type": "file_search",
        "vector_store_ids": [vector_store.id],
        "max_num_results": 20
    }],
    include = ["file_search_call.results"] # Include the references
)

In [66]:
# What is the answer?
Markdown(response.output[1].content[0].text)

Listen up, because I’m only going to say this once. If you want to understand the benefits of Bitte — the digital solution for restaurants — you better pay attention. Here’s the cold hard truth about what Bitte brings to the table, in terms only an investor obsessed with pre-revenue metrics and cash burn ratio would appreciate:

1. **Efficiency Boost & Staff Management**  
   - Digitalized, standardized processes slash the pain of high turnover. Fewer staff needed, higher efficiency. Reduced operational drag on your notoriously high cash burn ratio.  
   - Your floor team becomes a well-oiled machine with less time wasted on basic operational tasks.  

2. **Customer Experience on Steroids**  
   - Digital menus don’t mean less interaction — they mean smarter, personalized, attentive customer engagement. Staff are freed up to actually *wow* the customer, pushing satisfaction and loyalty through the roof.  
   - Visual menus, strategic pairing suggestions, and cross-selling features jack up your average ticket size. Expect a real bump in revenue, not just lip service.  

3. **Data-Driven Management**  
   - Real-time analytics tell you which dishes are cash cows and which are sinkholes. Identify bottlenecks in kitchen efficiency and optimize your menu accordingly.  
   - Intelligence-driven decisions slash waste and maximize profit margins. You’re not flying blind anymore — you’re running a tight ship with all data at your fingertips.

4. **Digital Innovation = Market Edge**  
   - Be where the customers are: visible on modern digital channels, offering delivery, takeaway, and online reservations through seamless integration.  
   - Innovation means survival. Being digital isn’t optional; it’s the only way to compete and grow without sinking more cash into an unsustainable burn.

5. **Easy to Adopt, Zero Hassle**  
   - No tech skills needed. They even set up your menu digitally from your PDF using OCR.  
   - No software installs, no crazy onboarding. You jump in, get your QR code, and start making more sales immediately.  

6. **Increase Your Top Line, Fast**  
   - Restaurants using digital menus see a 10-20% uptick in revenue versus paper menus. That’s not a guess — it’s backed by studies.  
   - Add multiple sales channels with delivery and takeaway by digital orders. More customers, more cash flowing in.

If you’re still worried about cash burn or pre-revenue phase, think of Bitte as your operational lever to maximize efficiency and monetize every interaction with your hungry customers. It’s not just a menu — it’s a profit multiplier and a survival tool for the brutal restaurant market out there.     